In [ ]:
import requests
import zipfile

url = "https://www.kaggle.com/api/v1/datasets/download/nirmalsankalana/plantdoc-dataset"
output_path = "plantdoc-dataset.zip"

# Télécharger le fichier
r = requests.get(url, stream=True)
with open(output_path, "wb") as f:
    for chunk in r.iter_content(chunk_size=8192):
        f.write(chunk)

# Décompresser
with zipfile.ZipFile(output_path, "r") as zip_ref:
    zip_ref.extractall("plantdoc")

print("Téléchargement et extraction terminés !")


In [ ]:
import os
from pathlib import Path
from collections import Counter
from PIL import Image
import matplotlib.pyplot as plt

def eda_dataset(dataset_path):
    dataset_path = Path(dataset_path)
    if not dataset_path.exists():
        print(f"Le chemin {dataset_path} n'existe pas !")
        return

    # Lister les classes
    classes = [d.name for d in dataset_path.iterdir() if d.is_dir()]
    print(f"Classes trouvées ({len(classes)}): {classes}\n")

    # Compter le nombre d'images par classe et stocker les dimensions
    image_counts = {}
    all_sizes = []

    for cls in classes:
        cls_path = dataset_path / cls
        # Ignorer fichiers/dossiers cachés
        images = [f for f in cls_path.glob("*.*") if not f.name.startswith(".")]
        image_counts[cls] = len(images)

        for img_file in images:
            try:
                with Image.open(img_file) as img:
                    all_sizes.append(img.size)
            except:
                pass

    # Afficher le nombre d'images par classe
    print(f"Nombre d'images par classe ({dataset_path.name}):")
    for cls, count in image_counts.items():
        print(f"{cls}: {count}")

    # Stats sur les dimensions
    if all_sizes:
        widths, heights = zip(*all_sizes)
        print(f"\nDimensions des images ({dataset_path.name}) :")
        print(f"Largeur : min={min(widths)}, max={max(widths)}, moyenne={sum(widths)//len(widths)}")
        print(f"Hauteur : min={min(heights)}, max={max(heights)}, moyenne={sum(heights)//len(heights)}")
    else:
        print("Aucune image valide trouvée.")

    # Visualiser quelques images par classe
    n_display = min(3, max(image_counts.values()))
    fig, axes = plt.subplots(len(classes), n_display, figsize=(12, len(classes)*3))
    for i, cls in enumerate(classes):
        cls_path = dataset_path / cls
        images = [f for f in cls_path.glob("*.*") if not f.name.startswith(".")][:n_display]
        for j, img_file in enumerate(images):
            with Image.open(img_file) as img:
                if len(classes) == 1:  # cas spécial si 1 seule classe
                    ax = axes[j]
                else:
                    ax = axes[i, j]
                ax.imshow(img)
                ax.set_title(cls)
                ax.axis('off')
    plt.tight_layout()
    plt.show()

    # Histogramme distribution des classes
    plt.figure(figsize=(14,6))
    plt.bar(image_counts.keys(), image_counts.values())
    plt.xticks(rotation=90)
    plt.title(f"Distribution des classes ({dataset_path.name})")
    plt.ylabel("Nombre d'images")
    plt.show()


# 🔹 Chemins vers train et test
train_path = "C:/Users/user/Desktop/computer_vision/plantdoc/train"
test_path  = "C:/Users/user/Desktop/computer_vision/plantdoc/test"

# Analyse train
print("===== ANALYSE TRAIN =====")
eda_dataset(train_path)

# Analyse test
print("\n===== ANALYSE TEST =====")
eda_dataset(test_path)


In [ ]:
import os
from pathlib import Path
from PIL import Image, ImageEnhance
import random
import shutil

# --- Fonctions d'augmentation ---
def augment_image(img):
    """Applique des transformations simples pour augmenter une image"""
    angle = random.choice([0, 90, 180, 270])
    img = img.rotate(angle)
    if random.random() > 0.5:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
    if random.random() > 0.5:
        img = img.transpose(Image.FLIP_TOP_BOTTOM)
    enhancer = ImageEnhance.Brightness(img)
    img = enhancer.enhance(random.uniform(0.7, 1.3))
    return img

# --- Créer dataset équilibré ---
def create_balanced_dataset(source_dir, target_dir):
    source_dir = Path(source_dir)
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)

    classes = [d.name for d in source_dir.iterdir() if d.is_dir()]
    print(f"Classes trouvées : {classes}")

    counts = {cls: len([f for f in (source_dir/cls).glob("*.*") if not f.name.startswith(".")]) for cls in classes}
    max_count = max(counts.values())
    print(f"Nombre max d'images pour équilibrage : {max_count}")

    for cls in classes:
        cls_source = source_dir / cls
        cls_target = target_dir / cls
        cls_target.mkdir(exist_ok=True)
        images = [f for f in cls_source.glob("*.*") if not f.name.startswith(".")]

        # Copier et convertir en RGB
        for img_file in images:
            try:
                with Image.open(img_file) as img:
                    if img.mode != "RGB":
                        img = img.convert("RGB")
                    img.save(cls_target / img_file.name)
            except Exception as e:
                print(f"Erreur copie {img_file}: {e}")

        # Générer images supplémentaires si nécessaire
        while len(list(cls_target.glob("*.*"))) < max_count:
            img_file = random.choice(images)
            try:
                with Image.open(img_file) as img:
                    img_aug = augment_image(img)
                    if img_aug.mode != "RGB":
                        img_aug = img_aug.convert("RGB")
                    new_name = f"{img_file.stem}_aug{random.randint(0,10000)}.jpg"
                    img_aug.save(cls_target / new_name)
            except Exception as e:
                print(f"Erreur augmentation {img_file}: {e}")

    print("✅ Dataset équilibré créé !")

# --- Créer dataset de validation ---
def create_val_dataset(train_dir, val_dir, val_ratio=0.2):
    train_dir = Path(train_dir)
    val_dir = Path(val_dir)
    val_dir.mkdir(parents=True, exist_ok=True)

    for cls_dir in train_dir.iterdir():
        if cls_dir.is_dir():
            cls_val_dir = val_dir / cls_dir.name
            cls_val_dir.mkdir(parents=True, exist_ok=True)

            images = [f for f in cls_dir.glob("*.*") if not f.name.startswith(".")]
            n_val = int(len(images) * val_ratio)
            val_images = random.sample(images, n_val)

            for img_file in val_images:
                shutil.move(str(img_file), cls_val_dir / img_file.name)

    print("✅ Dataset de validation créé !")

# --- Chemins ---
source_train = "C:/Users/user/Desktop/computer_vision/plantdoc/train"
balanced_train = "C:/Users/user/Desktop/computer_vision/plantdoc_balanced/train"
balanced_val = "C:/Users/user/Desktop/computer_vision/plantdoc_balanced/val"

# --- Exécution ---
create_balanced_dataset(source_train, balanced_train)
create_val_dataset(balanced_train, balanced_val, val_ratio=0.2)


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

def dataset_stats(dataset_path):
    dataset_path = Path(dataset_path)
    classes = [d.name for d in dataset_path.iterdir() if d.is_dir()]
    counts = {}

    total_images = 0
    for cls in classes:
        cls_path = dataset_path / cls
        n_images = len([f for f in cls_path.glob("*.*") if not f.name.startswith(".")])
        counts[cls] = n_images
        total_images += n_images

    print(f"Dataset : {dataset_path}")
    print(f"Nombre total d'images : {total_images}")
    print("Nombre d'images par classe :")
    for cls, n in counts.items():
        print(f"  {cls}: {n}")

    # Graphique
    plt.figure(figsize=(12,6))
    plt.bar(counts.keys(), counts.values())
    plt.xticks(rotation=90)
    plt.ylabel("Nombre d'images")
    plt.title(f"Distribution des images dans {dataset_path.name}")
    plt.show()

# --- Exemple d'utilisation ---
train_path = "C:/Users/user/Desktop/computer_vision/plantdoc_balanced/train"
val_path = "C:/Users/user/Desktop/computer_vision/plantdoc_balanced/val"

dataset_stats(train_path)
dataset_stats(val_path)


In [ ]:
import os
from pathlib import Path
from PIL import Image, ImageEnhance
import cv2
import numpy as np
import random
import shutil

# --- Préprocessing image ---
def preprocess_image(img_path, size=(224,224), apply_clahe=True, contrast_factor=1.2, brightness_factor=1.1):
    """
    Applique preprocessing à une image :
    - redimensionnement
    - conversion RGB
    - CLAHE sur canal L
    - ajustement contraste/luminosité
    """
    # Ouvrir image avec PIL
    img = Image.open(img_path)
    if img.mode != "RGB":
        img = img.convert("RGB")

    # Redimensionner
    img = img.resize(size)

    # Convertir en numpy pour OpenCV
    img_cv = np.array(img)

    if apply_clahe:
        # Convertir en LAB pour CLAHE
        lab = cv2.cvtColor(img_cv, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        l = clahe.apply(l)
        lab = cv2.merge((l, a, b))
        img_cv = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

    # Retour à PIL pour ajustement contraste/luminosité
    img = Image.fromarray(img_cv)
    enhancer_contrast = ImageEnhance.Contrast(img)
    img = enhancer_contrast.enhance(contrast_factor)
    enhancer_brightness = ImageEnhance.Brightness(img)
    img = enhancer_brightness.enhance(brightness_factor)

    return img

# --- Préparer dataset (train / val) ---
def preprocess_dataset(source_dir, target_dir, size=(224,224), apply_clahe=True):
    source_dir = Path(source_dir)
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)

    classes = [d.name for d in source_dir.iterdir() if d.is_dir()]
    print(f"Classes trouvées : {classes}")

    for cls in classes:
        cls_source = source_dir / cls
        cls_target = target_dir / cls
        cls_target.mkdir(exist_ok=True)

        images = [f for f in cls_source.glob("*.*") if not f.name.startswith(".")]

        for img_file in images:
            try:
                img = preprocess_image(img_file, size=size, apply_clahe=apply_clahe)
                img.save(cls_target / img_file.name)
            except Exception as e:
                print(f"Erreur traitement {img_file}: {e}")

    print(f"✅ Préprocessing terminé pour le dataset {source_dir}")

# --- Exemples de chemins ---
train_source = "C:/Users/user/Desktop/computer_vision/plantdoc_balanced/train"
val_source   = "C:/Users/user/Desktop/computer_vision/plantdoc_balanced/val"

train_target = "C:/Users/user/Desktop/computer_vision/plantdoc_preprocessed/train"
val_target   = "C:/Users/user/Desktop/computer_vision/plantdoc_preprocessed/val"

# --- Exécution ---
preprocess_dataset(train_source, train_target, size=(224,224), apply_clahe=True)
preprocess_dataset(val_source, val_target, size=(224,224), apply_clahe=True)
